# CodeDrift Arena GRPO Training (Colab)

This notebook provides a hackathon-ready run path:
install deps -> generate episodes -> train with GRPO -> save reward plot -> export adapter.

In [ ]:
# 1) Clone repo
!git clone https://github.com/bansalbhunesh/codedrift-arena.git
%cd codedrift-arena

In [ ]:
# 2) Install training dependencies
!pip install -r requirements-train.txt

In [ ]:
# 3) Run a baseline training job
!python training/train.py --episodes 300 --steps 200 --backend unsloth --output_dir codedrift_output

In [ ]:
# 4) Optional fallback if unsloth is unavailable in your runtime
# !python training/train.py --episodes 300 --steps 200 --backend hf --output_dir codedrift_output

In [ ]:
# 5) Plot reward curve (expects trainer history JSON/CSV exported by your run)
# Replace with your actual logging artifact path if needed.
import matplotlib.pyplot as plt
from pathlib import Path
import json

history_path = Path('codedrift_output') / 'trainer_state.json'
if history_path.exists():
    state = json.loads(history_path.read_text(encoding='utf-8'))
    log_hist = state.get('log_history', [])
    xs, ys = [], []
    for row in log_hist:
        if 'reward' in row and 'step' in row:
            xs.append(row['step'])
            ys.append(row['reward'])
    if xs:
        plt.figure(figsize=(8, 4))
        plt.plot(xs, ys)
        plt.title('CodeDrift GRPO Reward Curve')
        plt.xlabel('step')
        plt.ylabel('reward')
        plt.grid(alpha=0.3)
        Path('demo').mkdir(exist_ok=True)
        out = Path('demo') / 'training_reward_curve.png'
        plt.savefig(out, dpi=150, bbox_inches='tight')
        print(f'Saved plot to {out}')
    else:
        print('No reward rows found in trainer_state.json log_history')
else:
    print('trainer_state.json not found; inspect output dir and logging config')

In [ ]:
# 6) Quick verification artifact check
from pathlib import Path
print('Adapter exists:', (Path('codedrift_output') / 'final').exists())
print('Curve exists:', (Path('demo') / 'training_reward_curve.png').exists())

In [ ]:
# 7) V2 GRPO — pass every speed/throughput flag on one line (A100, etc.)
# Optional: `!wandb login` then add `--wandb` to log runs.
!python training_v2/train_v2.py --backend unsloth --episodes 100 --steps 50 --num_generations 2 --per_device_train_batch_size 1 --gradient_accumulation_steps 2 --learning_rate 5e-5 --warmup_steps 10 --logging_steps 20 --save_steps 500 --max_prompt_length 768 --max_completion_length 192 --temperature 0.8 --bf16 --no-curriculum --output_dir outputs/v2_fast

# V1 example with same style (add `--no_wandb` to skip W&B without editing code):
# !python training/train.py --backend unsloth --episodes 200 --steps 100 --num_generations 2 --gradient_accumulation_steps 2 --logging_steps 20 --no_wandb --output_dir codedrift_output